# QualityPhys - Camera Remote Vital Signs Estimator (CRVSE) Project

## Notebook P3-38: Three heart-rate checks - low rates, a browser-grade encode, lower frame rates

### What this notebook does
Tests three things about the app's heart-rate reading that the evaluations so far could not show, each under a rule fixed before it runs: whether the readout loses or misreads windows when the heart beats below 60 bpm; whether compressing a recording the way a browser recorder does moves the reading; and down to which frame rate the reading holds. Part 0 is the three rules, committed before any of the three runs, with the facts they rest on and a freeze record. The runs use only data already held - the Phase-3 stores and twelve raw VitalVideos-WW recordings - and the app's own estimator code; the later sections verify the outputs' provenance, apply the rules and state the outcomes. No verdict changes the app inside this notebook: a confirmed problem becomes a documented limitation, and any fix is a separate rule. No frame is rendered, and no recording appears beside its skin type, site or age.

### Estimators
- `hr_physnet_v2` - the shipped model and the app's default (NB_P3_18; weights SHA-256 f30fae25...e6b8). Every verdict is about this estimator.
- `hr_spectral` (POS) - the classical cross-check, reported beside v2 and deciding nothing.

`hr_physnet_v3` is not scored: it is a registered candidate, and it trained on 240 of VitalVideos-WW's 300 participants.

### Sets
- **VitalVideos-WW** - 300 recordings of 60 s, seated and still, lossless lab video at 60 fps, never used to train v2: its store for checks 1 and 3, and the twelve raw recordings kept (two per Fitzpatrick type) for check 2.
- **UBFC-Phys**, rest task (T1), and **Rice iPPG**, clamped crop - descriptive replications for check 1. The app's framing gate refuses every Rice recording, so Rice's are forced readings.

### Background
Every heart-rate evaluation so far read lossless lab video at 30 fps or more. A capture through the app differs in three ways the record has only touched: the browser compresses every frame, a phone may deliver fewer than 30 frames a second, and a person may rest below 60 bpm. Resting captures of one participant at about 50 bpm (2026-10-17) showed windows reading near twice the reference rate, and one refused capture that kept 2 of its 21 windows. The frame-rate gate (refuse below 20 fps, flag below 27) traces to a sweep on the Phase-2 model; the only Phase-3 evidence below 27 fps is five phone captures at 24 fps and an SQI-selected MCD subset.

## Part 0 - Pre-registration (2026-11-01)

Fixed before any of the three checks has run, and committed before the first run. Each check runs once. **Anything that changes after a check's run starts makes that check exploratory**, reported as such with the change and its reason. What was seen before these numbers were fixed is listed under *On record before the freeze*.

**Scope.** Checks of a method - the app's estimator code applied to stored crops and to re-encoded video - not of a capture through a browser. Retrospective data only: no new recording is made and nothing trains. Anything published is aggregates only.

### Common to the three checks
- **Estimators.** v2 decides; POS is reported beside it. Both run through `Estimator.estimate(frames, fps)`, the call the web app makes after `crops_from_video`.
- **The readout, as it stands** (the freeze record reads it from `config.py`). A 160-frame window every 80 frames. The spectral peak is sought in 0.66-3.0 Hz. A window's confidence is the power within one lobe of the peak - half-width the larger of fps/160 and 0.15 Hz, so 0.1875 Hz at 30 fps - over the power in the whole band. Windows on the band edge, below 0.65 confidence, or more than 3 MADs (floor 1 bpm) from the median are discarded; the harmonic guard is off. A recording is **reported** when its status carries a value (`ok`, `degraded_capture` or `unstable`), as in every benchmark script.
- **Outputs.** One script, `app/live_vitals/scripts/check_hr_robustness.py`, with one sub-command per check, writes to `Data/hr_robustness/`: summaries, committed, each carrying the provenance block every benchmark summary carries (command, commit, changed files, weights SHA-256, time); per-recording and per-window tables, kept local. The re-encoded videos are made by a standalone script outside the repository and never committed.
- **Intervals.** Where a check uses one: recordings resampled with replacement, all of a recording's windows together, 10,000 draws, seed 42 (`numpy.random.default_rng`), the 2.5th and 97.5th percentiles.
- **What a verdict can do.** Nothing in the app changes inside these checks. A confirmed problem is written up as a limitation in the README and the model card; a fix - a threshold, a readout change, re-recorded regression baselines - needs its own rule, written after the verdict, and its own commit.

### Check 1 - Low heart rates and the confidence gate

**Question.** Does the readout keep fewer usable, right windows when the heart beats below 60 bpm than at ordinary resting rates?

**Why it could.** Confidence divides the peak's power by the power in the whole 0.66-3.0 Hz band. A pulse's harmonics fall inside that band below 90 bpm (the second), below 60 bpm (the third as well) and below 45 bpm (the fourth), so the same waveform scores lower at a lower rate. Below about 51 bpm the lobe itself reaches the band's lower edge and is cut short. And a window whose second harmonic wins reads double.

**Data.** Window rows written at the run's commit by the benchmark scripts' own evaluation functions (`check_vitalvideos.evaluate`, `check_ubfc_phys.evaluate`, `check_rice_ippg.evaluate`), not reused from earlier runs - the VitalVideos-WW run on disk predates the provenance block. VitalVideos-WW, all 300 recordings, as stored (`frames`, 30 fps) - the decisive set. UBFC-Phys T1 and Rice's clamped crop - descriptive.

**Windows scored.** The benchmark scripts' *primary* windows: those the reference's own readout keeps, in recordings whose reference is reported. Each is classed by the reference's rate in that window: **low** below 60 bpm; **typical** 65-90 bpm, the gap keeping windows near the line out of both classes; **high** 90 bpm and above, reported and deciding nothing.

**Measure.** A window is **good** when v2's readout keeps it and its rate lies within 10% of the reference's. The decisive figure, on VitalVideos-WW: the share of good windows among low windows minus that among typical windows, in points, with the interval of *Common*.

**Outcomes** - the bar is a loss of 10 points:
- *confirmed* (low rates lose materially) when the interval's upper bound is at or below -10 points;
- *not confirmed* when its lower bound is above -10 points;
- *inconclusive* otherwise, reported as such and never reframed.

**Described beside it, deciding nothing.** Per class: the share kept; the share kept but more than 10% off; the share reading near twice the reference (within 10% of twice its rate), kept or not; the median window confidence of v2, of POS and of the contact PPG itself - the reference's own confidence shows whether the formula alone penalises a slow pulse, whatever the camera does. The same tables for UBFC-Phys T1 and Rice, and recording-level coverage and error by the recording's reference rate.

### Check 2 - A browser-grade encode

**Question.** Does compressing a recording the way a browser recorder does move the reading, against the same recording uncompressed?

**Data.** The twelve raw VitalVideos-WW recordings kept outside the store: two per Fitzpatrick type 1-6, from all three site groups; 960x1056 at 60 fps, 3600 frames, H.264 High 4:4:4 lossless.

**The ladder.** Every rung starts from the same frames: every second frame of the original (frames 0, 2, 4 ...: 1800 frames at 30 fps, the frames the store keeps), each scaled to half size (480x528) with `INTER_AREA`, plane by plane in the source's own YUV 4:4:4, so that the face spans about as many pixels as in a laptop webcam's 640x480 frame and the baseline stays lossless to the bit. Then:
- **R0** - lossless, 4:4:4 (libx264, quantiser 0): the baseline.
- **R1** - lossless, 4:2:0 (the chroma planes halved by the encoder's own conversion): the chroma subsampling every browser encode adds, alone.
- **R2** - H.264, 4:2:0, at 0.651 bits per pixel per frame, the app's recorder setting (640x480 at 6 Mbit/s and 30 fps), here 4.95 Mbit/s: **the decisive rung**.
- **R3** and **R4** - H.264 at half and a quarter of that, 0.326 and 0.163 bits per pixel per frame (2.48 and 1.24 Mbit/s). Live captures put the floor between 0.1 and 0.2.
- **R5** - VP8, the codec of a WebM recording, 4:2:0, at 0.651 bits per pixel per frame.

Lossy rungs imitate a real-time encoder: libx264 with preset `veryfast` and `tune zerolatency` (no B-frames, no look-ahead), the target as both average and maximum rate with a one-second buffer, a key frame every 60 frames; VP8 with `deadline realtime` and `cpu-used 4`, the same target and key-frame interval. PyAV 19.0.1 (libx264, libvpx). These settings imitate a browser's encoder and are not it; the conclusions say so.

**Procedure.** Each file is read as the web app reads an upload: `crops_from_video` (its own face detection, frozen box and capture gates), then v2 and POS. Every rung carries the same aspect flag - the frame is taller than wide - so the comparison stays paired.

**Measure.** Per recording, R2 against R0: the change in the reported value (bpm), and whether the recording changes between reported and not reported.

**Outcomes:**
- *no effect* when every recording R0 reports is also reported at R2, within 1.0 bpm of its R0 value;
- *effect* when at least 2 of the 12 lose their reading at R2 or move by more than 2.0 bpm;
- *inconclusive* otherwise.

The other rungs are judged the same way against R0 and decide nothing: R1 shows what chroma subsampling alone does, R3 and R4 where the bit rate starts to cost, R5 the other codec.

### Check 3 - Lower frame rates

**Question.** Down to which frame rate does v2's reading hold?

**Data.** All 300 VitalVideos-WW recordings, from the store's 60 fps crops (`frames_60`, 3600 frames) and the reference on the same grid (`bvp_60`).

**Thinning.** For each rate r of 30, 27, 25, 24, 22 and 20 fps: the frames nearest a uniform grid at r - index round(k x 60 / r) for k = 0, 1, 2 ... while it stays inside the recording - so a frame's time is off the grid by at most 8.3 ms, as a camera dropping frames would leave it. The reference is read at the same indices. At 30 fps the thinning returns exactly the store's `frames` (every second frame). The readout is told the rate r, as the app tells it a capture's measured rate; its windows stay 160 frames, so they last longer at lower rates (6.7 s at 24 fps, 8.0 s at 20 fps), as in a real capture.

**Measure.** Per recording and rate, the absolute error of the reported value against the reference read through the same windows. For each rate below 30, over the recordings reported at both rates: the mean of (absolute error at r) minus (absolute error at 30 fps), with the interval of *Common*; and the number of recordings reported at r against at 30 fps. The frame-rate gate is not applied, since it is what is being measured; every other status stays as the readout sets it.

**Outcomes, per rate:**
- *holds* when the interval's upper bound is at or below +0.5 bpm and at most 3 fewer recordings are reported than at 30 fps;
- *fails* when its lower bound is above +0.5 bpm, or more than 15 fewer are reported;
- *inconclusive* otherwise.

The verdict is the lowest rate that holds with every rate above it holding too. POS is judged the same way and decides nothing.

### Hypotheses, stated in advance
Judged on the outcomes above, with the point estimates beside them.
- **H1.** Check 1 is *not confirmed* for v2: on lab video, low rates cost it fewer than 10 points of good windows (moderate confidence).
- **H2.** The contact PPG's own median window confidence is at least 0.05 lower among low windows than among typical ones: the formula penalises a slow pulse whatever the camera does (high confidence).
- **H3.** v2's windows reading near twice the reference are at least twice as common among low windows as among typical ones (moderate confidence).
- **H4.** Check 2: *no effect* at R2, and an *effect* at R4 (moderate confidence).
- **H5.** Chroma subsampling alone (R1) moves no reading by more than 0.5 bpm (moderate confidence).
- **H6.** Check 3: v2 holds down to 24 fps and fails at 20 fps; POS holds at every rate (moderate confidence).

### Preflight
A check's outputs are scored only if every item passes for it; any failure stops the notebook.
1. Every output names one commit with no uncommitted app code, and between Part 0's commit and that commit nothing under `app/` changes except `check_hr_robustness.py` and its tests.
2. v2's weights are the registered file (SHA-256 f30fae25...e6b8).
3. No `--limit`: check 1 on every recording of its three stores, check 2 on all twelve recordings and every rung, check 3 on all 300 recordings at all six rates.
4. The settings each output records equal Part 0's.
5. Self-tests come first: check 3's 30 fps thinning equals the store's `frames` byte for byte in every recording; check 2's R0 decodes to exactly the scaled YUV planes it was made from, in every recording.

### Interpretation fixed in advance
- **Check 1.** *Confirmed:* the README and the model card state that below 60 bpm the app keeps fewer usable windows, with the figures; a fix to the confidence (counting the harmonics with the peak, or a band that follows the rate) is designed later under its own rule and judged on data other than this check's decisive set. *Not confirmed:* nothing changes, and the resting captures that raised the question stay an observation. *Inconclusive:* reported as such, and no fix rests on it.
- **Check 2.** *No effect:* the heart-rate figures' "lossless lab video" caveat can add that a browser-like encode of twelve such recordings moved no reading by more than 1 bpm - a proxy encoder, not a browser. *Effect:* the README states the bit rate, in bits per pixel per frame, at which readings degrade, and the app's recorder setting is reviewed.
- **Check 3.** The lowest rate that holds is evidence for a later decision on the 27 and 20 fps gates; moving either needs its own rule, re-recorded regression baselines and a look at phone captures. This check moves no gate.

### On record before the freeze

**The readout at Part 0's commit** (`config.py`; the freeze record checks each value): `CLIP_LEN` 160, `WINDOW_STRIDE` 80, `TARGET_FPS` 30; search band 0.66-3.0 Hz; `CONFIDENCE_FLOOR_HZ` 0.15; `MIN_CONFIDENCE` 0.65; `MAD_OUTLIER_K` 3 with `MAD_FLOOR_BPM` 1; `MIN_USABLE_WINDOWS` 3; `MIN_USABLE_FRACTION` 0.50 (below it `degraded_capture`); `MIN_REPORTABLE_FRACTION` 0.25; `HARMONIC_GUARD_ENABLED` false; frame-rate gates `MIN_FPS_ACCEPT` 27, `MIN_FPS_WARN` 20, `MAX_FPS_ACCEPT` 33. v2's weights: `phase_3_physnet_v2_baseline_best.zip`, 3,106,459 bytes.

**The data held.** `vitalvideos_phase3_rc_none.h5` (34.22 GB): 300 groups, each with 1800-frame `frames` at 30 fps and 3600-frame `frames_60` and `bvp_60` at 60 fps. `ubfc_phys_phase3_rc_none.h5` and `rice_ippg_phase3_rc_none.h5` as the benchmark scripts read them. The twelve raw recordings: 50.3 GB, two per Fitzpatrick type, five from MOL/RING, three from CRC/SCH/BET and four from CH*; each 960x1056, 60 fps, 3600 frames, H.264 High 4:4:4 lossless. PyAV 19.0.1 with libx264 and libvpx; no ffmpeg is installed.

**Seen before the freeze.**
- VitalVideos-WW, v2's zero-shot run of 2026-09-17, by each recording's reference rate (recordings / reported / `ok` / median absolute error of the reported, bpm): 55 bpm or below 6 / 5 / 3 / 0.70; 55-60 bpm 17 / 14 / 14 / 0.36; 60-65 bpm 41 / 33 / 31 / 0.26; 65-75 bpm 77 / 73 / 69 / 0.25; 75-90 bpm 102 / 99 / 88 / 0.31; above 90 bpm 53 / 48 / 44 / 0.26.
- In the earlier window files, the number of windows with a reference below 60 bpm: VitalVideos-WW 531 in 50 recordings, UBFC-Phys T1 368 in 38, Rice 404 in 23. Counts only: no window's confidence, kept flag or error was examined.
- The resting captures at about 50 bpm described in *Background*.
- Frame rate: five phone captures at 24 fps (two participants; MAE 1.04 bpm against an oximeter); the MCD subset's bias at 24 fps (-0.19 bpm, SQI-selected); UBFC-rPPG subject 25, a 23.2 fps recording that both estimators read within 1.5 bpm; the Phase-2 sweep of 2026-07-21 (good tracking needs 28-30 Hz, 22-26 Hz unreliable), on a different model.
- Compression: the capture study's bit-rate floor, abrupt between 3 and 6 Mbit/s at 720p (about 0.1-0.2 bits per pixel per frame), on live captures.
- No encode of the raw recordings and no thinned rate has been run.

### Freeze record
Checks that the code at this commit carries the readout written above and v2's registered weights, and that the data the checks will read is what Part 0 describes: the three stores present; every VitalVideos-WW group holding 1800-frame `frames` and 3600-frame `frames_60` and `bvp_60`, at 30 and 60 fps; the twelve raw recordings in their stated format. It records a fingerprint of the raw recordings - the SHA-256 of their sorted names and sizes - so that the run can show it read the same files. The cell stops if any check fails.

In [1]:
import hashlib
import sys
from pathlib import Path

import av
import h5py


def find_repo_root(start):
    '''The first folder at or above `start` holding both app/ and Data/.'''
    for folder in [start, *start.parents]:
        if (folder / 'app').is_dir() and (folder / 'Data').is_dir():
            return folder
    raise FileNotFoundError(f'no repository root at or above {start}')


REPO_ROOT = find_repo_root(Path.cwd().resolve())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from app.live_vitals import config
from app.live_vitals.scripts import check_rice_ippg, check_ubfc_phys, check_vitalvideos

# Part 0's values as written above. Each must equal the code at this commit.
FROZEN = {
    'CLIP_LEN': 160,
    'WINDOW_STRIDE': 80,
    'TARGET_FPS': 30.0,
    'HR_LOW_HZ': 0.66,
    'HR_HIGH_HZ': 3.0,
    'CONFIDENCE_FLOOR_HZ': 0.15,
    'MIN_CONFIDENCE': 0.65,
    'MAD_OUTLIER_K': 3.0,
    'MAD_FLOOR_BPM': 1.0,
    'MIN_USABLE_WINDOWS': 3,
    'MIN_USABLE_FRACTION': 0.50,
    'MIN_REPORTABLE_FRACTION': 0.25,
    'HARMONIC_GUARD_ENABLED': False,
    'MIN_FPS_ACCEPT': 27.0,
    'MIN_FPS_WARN': 20.0,
    'MAX_FPS_ACCEPT': 33.0,
}
V2_SHA256 = 'f30fae250fb948e2aaece9aae5ec2d77fcc438d7ef1abbe54ae878a7eecee6b8'
RAW_SUBSET = Path(r'D:\QualityPhys\demo_data\vitalvideos_subset')

problems = []
for name, expected in FROZEN.items():
    actual = getattr(config, name)
    if actual != expected:
        problems.append(f'config.{name} is {actual!r}, Part 0 says {expected!r}')

weights = Path(config.PHYSNET_CKPT)
weights_sha = hashlib.sha256(weights.read_bytes()).hexdigest()
if weights_sha != V2_SHA256:
    problems.append(f'v2 weights {weights.name} have SHA-256 {weights_sha}')

stores = {'VitalVideos-WW': check_vitalvideos.DEFAULT_STORE,
          'UBFC-Phys': check_ubfc_phys.DEFAULT_STORE,
          'Rice iPPG': check_rice_ippg.DEFAULT_STORE}
for label, path in stores.items():
    if not Path(path).is_file():
        problems.append(f'{label} store missing: {path}')

n_groups = 0
if Path(stores['VitalVideos-WW']).is_file():
    with h5py.File(stores['VitalVideos-WW'], 'r') as store:
        n_groups = len(store)
        for name in store:
            group = store[name]
            shapes = {key: group[key].shape for key in ('frames', 'frames_60', 'bvp_60')}
            rates = (float(group.attrs['fps']), float(group.attrs['fps_native']))
            if (shapes != {'frames': (1800, 72, 72, 3), 'frames_60': (3600, 72, 72, 3),
                           'bvp_60': (3600,)} or rates != (30.0, 60.0)):
                problems.append(f'VitalVideos-WW group {name}: {shapes}, fps {rates}')
    if n_groups != 300:
        problems.append(f'VitalVideos-WW store holds {n_groups} groups, not 300')

videos = sorted(RAW_SUBSET.glob('*/*_1.mp4'))
if len(videos) != 12:
    problems.append(f'{len(videos)} raw recordings under {RAW_SUBSET}, not 12')
for video in videos:
    with av.open(str(video)) as container:
        stream = container.streams.video[0]
        found = (stream.codec_context.name, stream.codec_context.pix_fmt, stream.codec_context.width,
                 stream.codec_context.height, float(stream.average_rate), stream.frames)
    if found != ('h264', 'yuv444p', 960, 1056, 60.0, 3600):
        problems.append(f'{video.parent.name}: {found}')
listing = ''.join(f'{video.name} {video.stat().st_size}\n' for video in videos)
fingerprint = hashlib.sha256(listing.encode('utf-8')).hexdigest()

if problems:
    raise SystemExit('Freeze record FAILED:\n  ' + '\n  '.join(problems))
print('Freeze record - Part 0 of NB_P3_38')
print(f'  readout settings  : {len(FROZEN)} checked against config.py, all equal')
print(f'  v2 weights        : {weights.name}, SHA-256 {weights_sha[:8]}...{weights_sha[-4:]}')
for label, path in stores.items():
    print(f'  {label:<17} : {Path(path).name}, {Path(path).stat().st_size / 1e9:.2f} GB')
print(f'  store contents    : {n_groups} groups, frames 1800 @ 30 fps, frames_60 and bvp_60 3600 @ 60 fps')
print(f'  raw recordings    : {len(videos)} x h264 4:4:4, 960x1056, 60 fps, 3600 frames, '
      f'{sum(v.stat().st_size for v in videos) / 1e9:.1f} GB')
print(f'  raw fingerprint   : {fingerprint}')

Freeze record - Part 0 of NB_P3_38
  readout settings  : 16 checked against config.py, all equal
  v2 weights        : phase_3_physnet_v2_baseline_best.zip, SHA-256 f30fae25...e6b8
  VitalVideos-WW    : vitalvideos_phase3_rc_none.h5, 34.22 GB
  UBFC-Phys         : ubfc_phys_phase3_rc_none.h5, 16.56 GB
  Rice iPPG         : rice_ippg_phase3_rc_none.h5, 7.97 GB
  store contents    : 300 groups, frames 1800 @ 30 fps, frames_60 and bvp_60 3600 @ 60 fps
  raw recordings    : 12 x h264 4:4:4, 960x1056, 60 fps, 3600 frames, 50.3 GB
  raw fingerprint   : bf9ef4370bc2f755b82e69052bb2b5d86432f621f0639b7ab75b5ecd340ee013


## 1. Check 1 - inputs and provenance
Part 0's preflight for check 1. The summary and tables in `Data/hr_robustness/` are used only if they name one commit that follows Part 0's, with no uncommitted app code and no app code changed since Part 0 but this check's script and tests; v2's registered weights; every recording of the three stores, the same recordings for both estimators, with no `--limit`; and settings equal to Part 0's, the readout's included. Any failure stops the notebook.

In [2]:
import json
import math
import subprocess
import sys
from pathlib import Path

import h5py
import pandas as pd


def find_repo_root(start):
    '''The first folder at or above `start` holding both app/ and Data/.'''
    for folder in [start, *start.parents]:
        if (folder / 'app').is_dir() and (folder / 'Data').is_dir():
            return folder
    raise FileNotFoundError(f'no repository root at or above {start}')


REPO_ROOT = find_repo_root(Path.cwd().resolve())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from app.live_vitals.scripts import check_hr_robustness as robust
from app.live_vitals.scripts import check_rice_ippg, check_ubfc_phys, check_vitalvideos

OUT = REPO_ROOT / 'Data' / 'hr_robustness'
PART0_COMMIT = '1e34db5'
ALLOWED_SINCE_PART0 = {'app/live_vitals/scripts/check_hr_robustness.py',
                       'app/live_vitals/tests/test_hr_robustness.py'}
V2_SHA256 = 'f30fae250fb948e2aaece9aae5ec2d77fcc438d7ef1abbe54ae878a7eecee6b8'

# Part 0's numbers for check 1; the settings the run records must equal them.
PART0_CHECK1 = {
    'classes': {'low_below': 60.0, 'typical': [65.0, 90.0], 'high_from': 90.0},
    'tolerance': 0.10, 'double_tolerance': 0.10, 'bar_points': -10.0,
    'interval': {'unit': 'recording', 'draws': 10000, 'seed': 42, 'percentiles': [2.5, 97.5]},
    'estimators': ['hr_physnet_v2', 'hr_spectral'], 'deciding': 'hr_physnet_v2',
    'reported_statuses': ['ok', 'degraded_capture', 'unstable'], 'limit': None,
}
PART0_SETS = [('VitalVideos-WW', 'decisive', check_vitalvideos.DEFAULT_STORE, 'vitalvideos'),
              ('UBFC-Phys T1', 'descriptive', check_ubfc_phys.DEFAULT_STORE, 'ubfc_phys_t1'),
              ('Rice iPPG, clamped crop', 'descriptive', check_rice_ippg.DEFAULT_STORE,
               'rice_clamped')]
# The readout as Part 0's freeze record found it.
PART0_READOUT = {
    'CLIP_LEN': 160, 'WINDOW_STRIDE': 80, 'TARGET_FPS': 30.0, 'HR_LOW_HZ': 0.66,
    'HR_HIGH_HZ': 3.0, 'CONFIDENCE_FLOOR_HZ': 0.15, 'MIN_CONFIDENCE': 0.65,
    'MAD_OUTLIER_K': 3.0, 'MAD_FLOOR_BPM': 1.0, 'MIN_USABLE_WINDOWS': 3,
    'MIN_USABLE_FRACTION': 0.50, 'MIN_REPORTABLE_FRACTION': 0.25,
    'HARMONIC_GUARD_ENABLED': False, 'MIN_FPS_ACCEPT': 27.0, 'MIN_FPS_WARN': 20.0,
    'MAX_FPS_ACCEPT': 33.0,
}


def git(*args):
    return subprocess.run(['git', *args], cwd=REPO_ROOT, capture_output=True, text=True)


def store_recordings(path, keep):
    '''The names of a store's recordings whose attributes `keep` accepts.'''
    with h5py.File(path, 'r') as store:
        return sorted(name for name in store if keep(store[name].attrs))


summary_path = OUT / 'summary_low_hr.json'
if not summary_path.is_file():
    raise SystemExit(f'Preflight FAILED: no check 1 summary at {summary_path}')
summary = json.loads(summary_path.read_text(encoding='utf-8'))
problems = []

# 1. One commit after Part 0's; no uncommitted app code; no app code changed since Part 0
#    but this check's script and tests.
state = (summary.get('provenance') or {}).get('git') or {}
commit = state.get('commit')
changed = set()
if not commit:
    problems.append('the summary names no commit')
else:
    if git('merge-base', '--is-ancestor', PART0_COMMIT, commit).returncode != 0:
        problems.append(f'the run commit {commit[:12]} does not follow Part 0 ({PART0_COMMIT})')
    # Git's status leaves untracked files out, so a run made before the script was
    # committed would look clean; the script has to be in the run's commit.
    if git('cat-file', '-e', f'{commit}:app/live_vitals/scripts/check_hr_robustness.py').returncode != 0:
        problems.append(f'the run commit {commit[:12]} does not hold check_hr_robustness.py, '
                        'so the code that ran is not on record')
    changed = set(git('diff', '--name-only', PART0_COMMIT, commit, '--', 'app').stdout.splitlines())
    if changed - ALLOWED_SINCE_PART0:
        problems.append(f'app code changed since Part 0: {sorted(changed - ALLOWED_SINCE_PART0)}')
    uncommitted = [path for path in state.get('changed', []) if path.startswith('app/')]
    if uncommitted:
        problems.append(f'uncommitted app code at the run: {uncommitted}')
    elif state.get('dirty') and len(state.get('changed', [])) >= 20:
        problems.append('the run lists 20 or more changed files, so its app code cannot be shown clean')

# 2. v2's registered weights.
weights = (summary.get('provenance') or {}).get('weights') or {}
if weights.get('sha256') != V2_SHA256:
    problems.append(f"v2's weights in the provenance are {weights.get('sha256')}")

# 3. No --limit: every recording of the three stores, the same for both estimators.
expected = {
    'vitalvideos': store_recordings(check_vitalvideos.DEFAULT_STORE, lambda attrs: True),
    'ubfc_phys_t1': store_recordings(check_ubfc_phys.DEFAULT_STORE,
                                     lambda attrs: str(attrs['task']) == 'T1'),
    'rice_clamped': store_recordings(check_rice_ippg.DEFAULT_STORE,
                                     lambda attrs: str(attrs['activity']) in check_rice_ippg.ACTIVITIES),
}
for name, role, store, key in PART0_SETS:
    for estimator in PART0_CHECK1['estimators']:
        table = OUT / f'recordings_low_hr_{key}_{estimator}.csv'
        if not table.is_file():
            problems.append(f'missing {table.name}')
            continue
        found = sorted(pd.read_csv(table, dtype={'recording': str})['recording'])
        if found != expected[key]:
            problems.append(f'{table.name}: {len(found)} recordings, not the store\'s {len(expected[key])}')

# 4. The settings the run records equal Part 0's, the readout's included.
settings = summary.get('settings', {})
for key, value in PART0_CHECK1.items():
    if settings.get(key) != value:
        problems.append(f'setting {key} is {settings.get(key)!r}, Part 0 says {value!r}')
recorded_sets = [(s['name'], s['role'], s['store']) for s in settings.get('sets', [])]
if recorded_sets != [(name, role, str(store)) for name, role, store, _ in PART0_SETS]:
    problems.append(f'sets, roles or stores differ from Part 0: {recorded_sets}')
if summary.get('readout') != PART0_READOUT:
    problems.append(f"the readout differs from the freeze record's: {summary.get('readout')}")

if problems:
    raise SystemExit('Preflight FAILED:\n  ' + '\n  '.join(problems))
print('Preflight - check 1: every item passes')
print(f"  {'run commit':<24}: {commit[:12]}, follows Part 0 ({PART0_COMMIT}); app files changed since: "
      f"{', '.join(sorted(changed)) or 'none'}")
print(f"  {'written':<24}: {summary['provenance']['written_utc']}")
print(f"  {'v2 weights':<24}: SHA-256 {V2_SHA256[:8]}...{V2_SHA256[-4:]}")
for name, role, store, key in PART0_SETS:
    print(f'  {name:<24}: {len(expected[key])} recordings, both estimators ({role})')
print(f"  {'settings':<24}: equal to Part 0's, the readout's {len(PART0_READOUT)} values included")

Preflight - check 1: every item passes
  run commit              : aa2f51a0a77d, follows Part 0 (1e34db5); app files changed since: app/live_vitals/scripts/check_hr_robustness.py, app/live_vitals/tests/test_hr_robustness.py
  written                 : 2026-11-01T10:57:59+00:00
  v2 weights              : SHA-256 f30fae25...e6b8
  VitalVideos-WW          : 300 recordings, both estimators (decisive)
  UBFC-Phys T1            : 56 recordings, both estimators (descriptive)
  Rice iPPG, clamped crop : 70 recordings, both estimators (descriptive)
  settings                : equal to Part 0's, the readout's 16 values included


## 2. Check 1 - the rule and the outcome
The decisive figure recomputed from the VitalVideos-WW window table with the script's own functions, and required to equal the summary's: v2's share of good windows among low windows minus that among typical ones, its 95% interval over recordings, and the outcome against the 10-point bar. Then H1-H3, judged on their point estimates.

In [3]:
def same(a, b):
    '''Equal, to rounding, or both missing.'''
    if a is None or b is None:
        return a is None and b is None
    return math.isclose(a, b, rel_tol=0.0, abs_tol=1e-9)


windows = pd.read_csv(OUT / 'windows_low_hr_vitalvideos_hr_physnet_v2.csv', dtype={'recording': str})
frame = robust.scored_windows(windows.to_dict('records'))
difference = robust.good_share_difference(frame)
outcome = robust.outcome(difference['lower'], difference['upper'])
recorded = summary['sets']['VitalVideos-WW']['estimators']['hr_physnet_v2']
mismatches = [key for key in ('point', 'lower', 'upper')
              if not same(difference[key], recorded['low_minus_typical_points'][key])]
mismatches += [key for key in ('recordings', 'draws', 'draws_used')
               if difference[key] != recorded['low_minus_typical_points'][key]]
if outcome != recorded['outcome'] or outcome != summary['verdict']['outcome']:
    mismatches.append('outcome')
if mismatches:
    raise SystemExit(f'The window table does not reproduce the summary: {mismatches}')

classes = robust.class_figures(frame)
low, typical = classes['low'], classes['typical']


def share(value):
    return '-' if value is None else f'{100 * value:.1f}%'


print('Check 1 - VitalVideos-WW, PhysNet v2, primary windows (recomputed from the table)')
print(f"  low, below 60 bpm    : {low['windows']} windows in {low['recordings']} recordings, "
      f"good {share(low['share_good'])}")
print(f"  typical, 65-90 bpm   : {typical['windows']} windows in {typical['recordings']} recordings, "
      f"good {share(typical['share_good'])}")
if difference['lower'] is None:
    print('  low minus typical    : not estimable')
else:
    print(f"  low minus typical    : {difference['point']:+.1f} points (95% {difference['lower']:+.1f} "
          f"to {difference['upper']:+.1f}; {difference['recordings']} recordings, "
          f"{difference['draws_used']} of {difference['draws']} draws)")
print(f'  against the bar of {robust.BAR_POINTS:+.0f} points: {outcome.upper()}')

reference_gap = (None if None in (low['median_reference_confidence'],
                                  typical['median_reference_confidence'])
                 else typical['median_reference_confidence'] - low['median_reference_confidence'])
doubled_low, doubled_typical = low['share_doubled'], typical['share_doubled']
h1 = {'not confirmed': 'consistent', 'confirmed': 'contradicted'}.get(outcome, f'unresolved ({outcome})')
h2 = ('not estimable' if reference_gap is None
      else 'consistent' if reference_gap >= 0.05 else 'contradicted')
h3 = ('not estimable' if None in (doubled_low, doubled_typical)
      else 'consistent' if doubled_low > 0 and doubled_low >= 2 * doubled_typical else 'contradicted')
print('\nHypotheses, on point estimates')
print(f'  H1  check 1 not confirmed for v2: {h1}')
print(f"  H2  the contact PPG's median confidence, typical minus low: "
      f"{'-' if reference_gap is None else f'{reference_gap:+.3f}'} (0.05 or more): {h2}")
print(f'  H3  windows reading near twice the reference: low {share(doubled_low)}, '
      f'typical {share(doubled_typical)} (twice or more): {h3}')

Check 1 - VitalVideos-WW, PhysNet v2, primary windows (recomputed from the table)
  low, below 60 bpm    : 478 windows in 42 recordings, good 63.6%
  typical, 65-90 bpm   : 3687 windows in 221 recordings, good 85.8%
  low minus typical    : -22.2 points (95% -35.8 to -9.5; 250 recordings, 10000 of 10000 draws)
  against the bar of -10 points: INCONCLUSIVE

Hypotheses, on point estimates
  H1  check 1 not confirmed for v2: unresolved (inconclusive)
  H2  the contact PPG's median confidence, typical minus low: +0.036 (0.05 or more): contradicted
  H3  windows reading near twice the reference: low 8.6%, typical 0.3% (twice or more): consistent


## 3. Check 1 - description
Deciding nothing: for each set and estimator, the windows of each class with the shares kept, good, kept but off and reading near twice the reference, and the median confidences of the estimator and of the contact PPG itself; then coverage and error by the recording's reference rate. Aggregates only.

In [4]:
robust.print_low_hr(summary)

print("\nBy the recording's reference rate: recordings / reported / ok / median absolute error (bpm)")
for set_name, block in summary['sets'].items():
    for name, figures in block['estimators'].items():
        bands = '; '.join(
            f"{b['band']} {b['recordings']}/{b['reported']}/{b['ok']}/"
            f"{'-' if b['median_abs_error'] is None else format(b['median_abs_error'], '.2f')}"
            for b in figures['recording_bands'])
        print(f'  {set_name} | {name}: {bands}')


=== VitalVideos-WW (decisive) | hr_physnet_v2 | 6035 primary windows in 300 recordings
  class     windows  recs     kept   good    off double     conf    ref
  low           478    42    64.6%  63.6%   1.0%   8.6%    0.755  0.751
  gap           795    71    75.8%  74.0%   1.9%   2.8%    0.878  0.772
  typical      3687   221    86.2%  85.8%   0.4%   0.3%    0.894  0.787
  high         1075    70    81.9%  81.2%   0.7%   0.0%    0.893  0.914
  low - typical, good windows: -22.2 points (95% -35.8 to -9.5; 250 recordings, 10000 of 10000 draws) -> inconclusive

=== VitalVideos-WW (decisive) | hr_spectral | 6035 primary windows in 300 recordings
  class     windows  recs     kept   good    off double     conf    ref
  low           478    42    77.8%  77.2%   0.6%   1.5%    0.792  0.751
  gap           795    71    74.3%  74.1%   0.3%   2.4%    0.797  0.772
  typical      3687   221    69.4%  69.3%   0.1%   0.7%    0.768  0.787
  high         1075    70    64.0%  63.9%   0.1%   0.0%    0

## 4. Check 1 - findings (2026-11-01)

**The pre-registered test is inconclusive.** On VitalVideos-WW's primary windows, PhysNet v2 kept a reading within 10% of the reference in 63.6% of the windows whose reference beat below 60 bpm (478 windows in 42 recordings), against 85.8% between 65 and 90 bpm (3,687 windows in 221 recordings): a difference of -22.2 points, 95% interval -35.8 to -9.5 over recordings. The interval lies wholly below zero, yet its upper end misses the bar of -10 by 0.5 points, so a material loss is neither confirmed nor ruled out; as Part 0 fixed, the outcome is reported as such and no fix rests on it.

**The loss points to v2 rather than to the confidence formula.** The contact PPG's own median window confidence is only 0.036 lower among low windows than among typical ones (H2 asked for at least 0.05: contradicted), while v2's falls from 0.894 to 0.755. POS, read through the same formula, keeps more good windows at low rates than at typical ones (+7.9 points, 95% -5.6 to +19.7). The harmonics that the search band takes in below 90 bpm (the second) and below 60 bpm (the third) therefore do not explain v2's pattern on their own.

**Readings near twice the reference gather below 60 bpm** (H3: consistent): 8.6% of v2's low windows against 0.3% of its typical ones, and none above 90 bpm, where twice the rate leaves the search band. Most are discarded rather than reported: v2 keeps 64.6% of its low windows, and only 1.0% are kept more than 10% off, so the cost shows as fewer usable windows rather than as wrong readings. Per recording the same holds: at 55 bpm or below v2 reported 5 of 6 recordings, 3 of them `ok`, with a median absolute error of 0.70 bpm.

**The descriptive sets.** For v2 they point the same way: UBFC-Phys T1 -34.6 points (95% -62.9 to -12.8; 15 low recordings, a wrist reference) and Rice's clamped crop -26.5 (-43.6 to -5.6; 20 low recordings, forced readings). For POS they part: UBFC-Phys T1 -14.1 (-63.3 to +14.3), Rice +3.6 (-6.3 to +12.4). On UBFC-Phys both estimators also keep many readings more than 10% off the reference in every class (v2: 34.8% of low windows, 20.7% of typical ones). These sets decide nothing.

**What it leaves.** H1 is unresolved. Below 60 bpm the app's heart rate carries a described uncertainty: v2 keeps fewer usable windows there, and the failure seen in the windows is a reading near twice the rate - the pattern of the resting captures that raised the question. Any change to the readout would need its own rule, judged on data other than this check's decisive set.

## 5. Check 2 - inputs and provenance
Part 0's preflight for check 2. The summary and table in `Data/hr_robustness/` are used only if they name one commit that follows Part 0's and holds this check's script, with no uncommitted app code and no app code changed since Part 0 but this check's script and tests; v2's registered weights; the twelve raw recordings the freeze record fingerprinted, every one on every rung for both estimators, with no `--limit`; and settings equal to Part 0's - the ladder, the rule's numbers and the readout. The ladder must be the one its manifest records: the manifest the run read, holding the run's recipe, written at a commit that passes the same commit rules and precedes the run's. The self-test must have passed first - every R0 decodes to exactly the scaled planes it was made from - and each rung's files must be the codec and pixel format Part 0 names. Any failure stops the notebook.

Seen before the run, from the store's recorded face boxes and with no rate read: the framing gate passes 11 of the 12 recordings, with the aspect warning Part 0 expects, and refuses one, whose padded box loses 3.1% of its side outside the frame against a 2% limit. R0 is therefore expected to report at most 11 recordings; the rule counts only those it reports.

In [5]:
import hashlib

# Part 0's numbers for check 2; the settings the run records must equal them.
PART0_RAW_SUBSET = Path(r'D:\QualityPhys\demo_data\vitalvideos_subset')
PART0_RAW_FINGERPRINT = 'bf9ef4370bc2f755b82e69052bb2b5d86432f621f0639b7ab75b5ecd340ee013'
BROWSER_BITS = 6e6 / (640 * 480 * 30)  # the app's recorder: 640x480 at 6 Mbit/s and 30 fps
PART0_CHECK2 = {
    'source': {'codec': 'h264', 'pix_fmt': 'yuv444p', 'height': 1056, 'width': 960,
               'fps': 60.0, 'frames': 3600},
    'frame_step': 2,
    'ladder': {'height': 528, 'width': 480, 'fps': 30,
               'scaling': 'INTER_AREA, plane by plane in YUV 4:4:4'},
    'browser_bits_per_pixel_frame': BROWSER_BITS,
    'keyframe_interval': 60,
    'pyav': '19.0.1',
    'decisive_rung': 'R2', 'no_effect_bpm': 1.0, 'effect_bpm': 2.0, 'effect_recordings': 2,
    'estimators': ['hr_physnet_v2', 'hr_spectral'], 'deciding': 'hr_physnet_v2',
    'reported_statuses': ['ok', 'degraded_capture', 'unstable'],
    'recordings': 12, 'raw_subset': str(PART0_RAW_SUBSET), 'limit': None,
}
RECIPE_KEYS = ('source', 'frame_step', 'ladder', 'browser_bits_per_pixel_frame',
               'keyframe_interval', 'rungs', 'pyav')
LOSSLESS = {'qp': '0', 'preset': 'veryfast', 'g': '60'}


def real_time(bits, codec):
    '''A lossy rung's target in bits per second and Part 0's real-time encoder settings for it.'''
    rate = bits * 480 * 528 * 30
    cap = {'maxrate': str(int(rate)), 'bufsize': str(int(rate)), 'g': '60'}
    if codec == 'libx264':
        return rate, {'preset': 'veryfast', 'tune': 'zerolatency', **cap}
    return rate, {'deadline': 'realtime', 'cpu-used': '4', **cap}


PART0_RUNGS = [
    {'rung': 'R0', 'codec': 'libx264', 'container': 'mp4', 'pix_fmt': 'yuv444p', 'bits': None,
     'target_bit_rate': None, 'options': LOSSLESS},
    {'rung': 'R1', 'codec': 'libx264', 'container': 'mp4', 'pix_fmt': 'yuv420p', 'bits': None,
     'target_bit_rate': None, 'options': LOSSLESS},
]
for rung, codec, container, bits in (('R2', 'libx264', 'mp4', BROWSER_BITS),
                                     ('R3', 'libx264', 'mp4', BROWSER_BITS / 2),
                                     ('R4', 'libx264', 'mp4', BROWSER_BITS / 4),
                                     ('R5', 'libvpx', 'webm', BROWSER_BITS)):
    rate, options = real_time(bits, codec)
    PART0_RUNGS.append({'rung': rung, 'codec': codec, 'container': container, 'pix_fmt': 'yuv420p',
                        'bits': bits, 'target_bit_rate': rate, 'options': options})
RUNG_NAMES = [rung['rung'] for rung in PART0_RUNGS]
# What each rung's files decode as: codec and pixel format.
PART0_FILES = {'R0': ['h264', 'yuv444p'], 'R1': ['h264', 'yuv420p'], 'R2': ['h264', 'yuv420p'],
               'R3': ['h264', 'yuv420p'], 'R4': ['h264', 'yuv420p'], 'R5': ['vp8', 'yuv420p']}


def commit_problems(state, label):
    '''Part 0's first preflight item for one recorded git state, and the app files changed since Part 0.'''
    commit = (state or {}).get('commit')
    if not commit:
        return [f'{label} names no commit'], set()
    problems = []
    if git('merge-base', '--is-ancestor', PART0_COMMIT, commit).returncode != 0:
        problems.append(f'{label}: commit {commit[:12]} does not follow Part 0 ({PART0_COMMIT})')
    # Git's status leaves untracked files out, so the script has to be in the commit itself.
    if git('cat-file', '-e', f'{commit}:app/live_vitals/scripts/check_hr_robustness.py').returncode != 0:
        problems.append(f'{label}: commit {commit[:12]} does not hold check_hr_robustness.py')
    changed = set(git('diff', '--name-only', PART0_COMMIT, commit, '--', 'app').stdout.splitlines())
    if changed - ALLOWED_SINCE_PART0:
        problems.append(f'{label}: app code changed since Part 0: {sorted(changed - ALLOWED_SINCE_PART0)}')
    uncommitted = [path for path in state.get('changed', []) if path.startswith('app/')]
    if uncommitted:
        problems.append(f'{label}: uncommitted app code: {uncommitted}')
    elif state.get('dirty') and len(state.get('changed', [])) >= 20:
        problems.append(f'{label} lists 20 or more changed files, so its app code cannot be shown clean')
    return problems, changed


encode_path = OUT / 'summary_reencode.json'
if not encode_path.is_file():
    raise SystemExit(f'Preflight FAILED: no check 2 summary at {encode_path}')
encode = json.loads(encode_path.read_text(encoding='utf-8'))
settings = encode.get('settings', {})
problems = []

# 1. The run's commit: after Part 0's, holding the script, no other app code changed or uncommitted.
run_state = (encode.get('provenance') or {}).get('git') or {}
run_problems, run_changed = commit_problems(run_state, 'the run')
problems += run_problems

# 2. v2's registered weights.
weights = (encode.get('provenance') or {}).get('weights') or {}
if weights.get('sha256') != V2_SHA256:
    problems.append(f"v2's weights in the provenance are {weights.get('sha256')}")

# 3. No --limit: the twelve raw recordings Part 0 fingerprinted, each read on every rung by both
#    estimators, once.
raw = sorted(PART0_RAW_SUBSET.glob('*/*_1.mp4'))
listing = ''.join(f'{video.name} {video.stat().st_size}\n' for video in raw)
if hashlib.sha256(listing.encode('utf-8')).hexdigest() != PART0_RAW_FINGERPRINT:
    problems.append(f'the raw recordings under {PART0_RAW_SUBSET} are not the ones Part 0 fingerprinted')
names = [video.parent.name for video in raw]
table_path = OUT / 'recordings_reencode.csv'
if not table_path.is_file():
    raise SystemExit(f'Preflight FAILED: no check 2 table at {table_path}')
table = pd.read_csv(table_path, dtype={'recording': str})
keys = list(zip(table['recording'], table['rung'], table['estimator']))
wanted = {(name, rung, estimator) for name in names for rung in RUNG_NAMES
          for estimator in PART0_CHECK2['estimators']}
if len(keys) != len(wanted) or set(keys) != wanted:
    problems.append(f"the table holds {len(keys)} readings of {table['recording'].nunique()} recordings; "
                    f'Part 0 asks for {len(wanted)}, each of 12 recordings on 6 rungs by 2 estimators once')

# 4. The settings the run records equal Part 0's: the ladder, the rule's numbers, the readout.
for key, value in PART0_CHECK2.items():
    if settings.get(key) != value:
        problems.append(f'setting {key} is {settings.get(key)!r}, Part 0 says {value!r}')
if settings.get('rungs') != PART0_RUNGS:
    problems.append(f"the rungs differ from Part 0's: {settings.get('rungs')}")
if encode.get('readout') != PART0_READOUT:
    problems.append(f"the readout differs from the freeze record's: {encode.get('readout')}")

# 5. The ladder: the manifest the run read, holding the run's recipe, written at a commit that
#    passes item 1 too and precedes the run's.
ladder = encode.get('ladder') or {}
manifest_path = Path(settings.get('ladder_dir', '')) / 'manifest.json'
manifest, ladder_state = {}, {}
if not manifest_path.is_file():
    problems.append(f'no ladder manifest at {manifest_path}')
elif hashlib.sha256(manifest_path.read_bytes()).hexdigest() != ladder.get('manifest_sha256'):
    problems.append(f'{manifest_path} is not the manifest the run read')
else:
    manifest = json.loads(manifest_path.read_text(encoding='utf-8'))
    if manifest.get('recipe') != {key: settings.get(key) for key in RECIPE_KEYS}:
        problems.append("the manifest's recipe is not the run's")
    if sorted(manifest.get('recordings', [])) != names or len(manifest.get('files', {})) != 72:
        problems.append(f"the manifest lists {len(manifest.get('recordings', []))} recordings and "
                        f"{len(manifest.get('files', {}))} files, not the 12 and 72 of the ladder")
    ladder_state = manifest.get('repository') or {}
    problems += commit_problems(ladder_state, 'the ladder')[0]
    ladder_commit, run_commit = ladder_state.get('commit'), run_state.get('commit')
    if (ladder_commit and run_commit
            and git('merge-base', '--is-ancestor', ladder_commit, run_commit).returncode != 0):
        problems.append(f'the ladder commit {ladder_commit[:12]} does not precede the run commit {run_commit[:12]}')

# 6. The self-test came first and passed in every recording; each rung's files are what Part 0 names.
if encode.get('self_test') != {'recordings': 12, 'equal': 12, 'frames_each': [1800], 'max_abs_diff': 0}:
    problems.append(f"the self-test did not pass in every recording: {encode.get('self_test')}")
files = {rung: [block.get('codec'), block.get('pix_fmt')] for rung, block in encode.get('files', {}).items()}
if files != PART0_FILES:
    problems.append(f"the rungs' files are not the codecs and pixel formats Part 0 names: {files}")

if problems:
    raise SystemExit('Preflight FAILED:\n  ' + '\n  '.join(problems))
print('Preflight - check 2: every item passes')
print(f"  {'run commit':<24}: {run_state['commit'][:12]}, follows Part 0 ({PART0_COMMIT}); "
      f"app files changed since: {', '.join(sorted(run_changed)) or 'none'}")
print(f"  {'written':<24}: {encode['provenance']['written_utc']}")
print(f"  {'ladder':<24}: {len(manifest['files'])} files written {manifest['written_utc']} at "
      f"{ladder_state['commit'][:12]}; manifest SHA-256 {ladder['manifest_sha256'][:8]}...")
print(f"  {'v2 weights':<24}: SHA-256 {V2_SHA256[:8]}...{V2_SHA256[-4:]}")
print(f"  {'raw recordings':<24}: the 12 Part 0 fingerprinted ({PART0_RAW_FINGERPRINT[:8]}...)")
print(f"  {'readings':<24}: {len(keys)}, each of 12 recordings on 6 rungs by 2 estimators")
print(f"  {'self-test':<24}: every R0 decodes to its scaled source, 1800 frames each, largest difference 0")
print(f"  {'settings':<24}: equal to Part 0's, the six rungs and the readout's {len(PART0_READOUT)} values included")

Preflight - check 2: every item passes
  run commit              : 46a6cda300e1, follows Part 0 (1e34db5); app files changed since: app/live_vitals/scripts/check_hr_robustness.py, app/live_vitals/tests/test_hr_robustness.py
  written                 : 2026-11-01T13:49:30+00:00
  ladder                  : 72 files written 2026-11-01T12:15:07+00:00 at 41736314e7da; manifest SHA-256 e9300340...
  v2 weights              : SHA-256 f30fae25...e6b8
  raw recordings          : the 12 Part 0 fingerprinted (bf9ef437...)
  readings                : 144, each of 12 recordings on 6 rungs by 2 estimators
  self-test               : every R0 decodes to its scaled source, 1800 frames each, largest difference 0
  settings                : equal to Part 0's, the six rungs and the readout's 16 values included


## 6. Check 2 - the rule and the outcome
Every rung's outcome against R0, for both estimators, recomputed from the per-recording table with the script's own function and required to equal the summary's. For v2: the recordings R0 reports, those lost, those moved by more than 1.0 and 2.0 bpm, the largest move and the outcome; R2 decides. Then H4 and H5 on v2's outcomes; under H5 a reading lost at R1 counts against it.

In [6]:
def readings(estimator, rung):
    '''One estimator's readings at one rung, by recording, as rung_outcome takes them.'''
    part = table[(table['estimator'] == estimator) & (table['rung'] == rung)]
    return {row.recording: {'value': float(row.value), 'reported': bool(row.reported)}
            for row in part.itertuples()}


recomputed, mismatches = {}, []
for estimator in PART0_CHECK2['estimators']:
    base = readings(estimator, 'R0')
    for rung in RUNG_NAMES[1:]:
        result = robust.rung_outcome(base, readings(estimator, rung))
        recomputed[estimator, rung] = result
        recorded = encode['rungs'][rung][estimator]
        for key, value in result.items():
            numeric = value is None or isinstance(value, float)
            if (not same(value, recorded.get(key))) if numeric else value != recorded.get(key):
                mismatches.append(f'{rung} {estimator} {key}')
decisive = recomputed[PART0_CHECK2['deciding'], PART0_CHECK2['decisive_rung']]
if encode.get('verdict') != {'rung': PART0_CHECK2['decisive_rung'], 'estimator': PART0_CHECK2['deciding'],
                             'outcome': decisive['outcome']}:
    mismatches.append('verdict')
if mismatches:
    raise SystemExit(f'The table does not reproduce the summary: {mismatches}')

print('Check 2 - PhysNet v2, every rung against R0 (recomputed from the table)')
for rung in RUNG_NAMES[1:]:
    result = recomputed['hr_physnet_v2', rung]
    largest = '-' if result['max_move'] is None else f"{result['max_move']:.2f}"
    label = f'{rung} (decisive)' if rung == PART0_CHECK2['decisive_rung'] else rung
    print(f"  {label:<13}: R0 reports {result['base_reported']} of 12; lost {result['lost']}, "
          f"gained {result['gained']}; moved >1.0 bpm {result['moved_over_no_effect']}, "
          f">2.0 bpm {result['moved_over_effect']}; largest {largest} bpm -> {result['outcome'].upper()}")
print(f"\nR2 against R0, PhysNet v2: {decisive['outcome'].upper()}")


def holds(outcome, wanted, opposite):
    return 'holds' if outcome == wanted else 'fails' if outcome == opposite else 'unresolved'


r1, r2, r4 = (recomputed['hr_physnet_v2', rung] for rung in ('R1', 'R2', 'R4'))
parts = (holds(r2['outcome'], 'no effect', 'effect'), holds(r4['outcome'], 'effect', 'no effect'))
h4 = 'consistent' if parts == ('holds', 'holds') else 'contradicted' if 'fails' in parts else 'unresolved'
if r1['base_reported'] == 0:
    h5 = 'not estimable'
else:
    h5 = 'consistent' if r1['lost'] == 0 and r1['max_move'] <= 0.5 else 'contradicted'
print('\nHypotheses')
print(f"  H4  no effect at R2 and an effect at R4: R2 {r2['outcome']}, R4 {r4['outcome']}: {h4}")
print(f"  H5  chroma subsampling alone (R1) moves no reading by more than 0.5 bpm: lost {r1['lost']}, "
      f"largest move {'-' if r1['max_move'] is None else format(r1['max_move'], '.2f')} bpm: {h5}")

Check 2 - PhysNet v2, every rung against R0 (recomputed from the table)
  R1           : R0 reports 10 of 12; lost 0, gained 0; moved >1.0 bpm 0, >2.0 bpm 0; largest 0.43 bpm -> NO EFFECT
  R2 (decisive): R0 reports 10 of 12; lost 1, gained 0; moved >1.0 bpm 0, >2.0 bpm 0; largest 0.72 bpm -> INCONCLUSIVE
  R3           : R0 reports 10 of 12; lost 1, gained 0; moved >1.0 bpm 1, >2.0 bpm 0; largest 1.00 bpm -> INCONCLUSIVE
  R4           : R0 reports 10 of 12; lost 2, gained 0; moved >1.0 bpm 2, >2.0 bpm 1; largest 2.11 bpm -> EFFECT
  R5           : R0 reports 10 of 12; lost 2, gained 0; moved >1.0 bpm 1, >2.0 bpm 0; largest 1.38 bpm -> EFFECT

R2 against R0, PhysNet v2: INCONCLUSIVE

Hypotheses
  H4  no effect at R2 and an effect at R4: R2 inconclusive, R4 effect: unresolved
  H5  chroma subsampling alone (R1) moves no reading by more than 0.5 bpm: lost 0, largest move 0.43 bpm: consistent


## 7. Check 2 - description
Deciding nothing: what each rung's files hold against the target in bits per pixel per frame, and every rung's outcome for both estimators; then, by rung, the capture verdicts, the statuses, v2's moves against R0 in order of size, and each estimator's median absolute error against the ECG rate. Aggregates only.

In [8]:
robust.print_reencode(encode)

v2_rows = table[table['estimator'] == 'hr_physnet_v2']
print('\nCapture verdicts by rung (the capture path decides before either estimator)')
for rung in RUNG_NAMES:
    counts = v2_rows[v2_rows['rung'] == rung]['capture_verdict'].value_counts()
    print(f'  {rung}: ' + ', '.join(f'{verdict} {counts.get(verdict, 0)}'
                                   for verdict in ('ACCEPT', 'WARN', 'REJECT')))

print('\nStatuses by rung')
for estimator in PART0_CHECK2['estimators']:
    rows = table[table['estimator'] == estimator]
    for rung in RUNG_NAMES:
        counts = rows[rows['rung'] == rung]['status'].value_counts().sort_index()
        print(f'  {estimator:<14} {rung}: ' + ', '.join(f'{status} {n}' for status, n in counts.items()))

print("\nv2's moves against R0, bpm, recordings reported at both, smallest first")
for rung in RUNG_NAMES[1:]:
    moves = v2_rows[v2_rows['rung'] == rung]['change_vs_r0'].abs().dropna().sort_values()
    print(f'  {rung}: ' + (' '.join(f'{move:.2f}' for move in moves) or '-'))

print('\nMedian absolute error against the ECG rate (hr_ecg), bpm, readings reported')
for estimator, by_rung in encode['reference']['median_abs_error'].items():
    print(f'  {estimator:<14}: ' + '; '.join(
        f"{rung} {'-' if value is None else format(value, '.2f')}" for rung, value in by_rung.items()))


self-test: R0 decodes to its scaled source in 12 of 12 recordings ([1800] frames each; largest difference 0)
  R0: h264 yuv444p, median 8.652 bits per pixel per frame (target lossless)
  R1: h264 yuv420p, median 3.616 bits per pixel per frame (target lossless)
  R2: h264 yuv420p, median 0.656 bits per pixel per frame (target 0.651)
  R3: h264 yuv420p, median 0.328 bits per pixel per frame (target 0.326)
  R4: h264 yuv420p, median 0.164 bits per pixel per frame (target 0.163)
  R5: vp8 yuv420p, median 0.662 bits per pixel per frame (target 0.651)
  R1 | hr_physnet_v2 : R0 reports 10, lost 0, gained 0, moved >1 bpm 0, >2 bpm 0, largest 0.43 -> no effect
  R1 | hr_spectral   : R0 reports 10, lost 0, gained 0, moved >1 bpm 0, >2 bpm 0, largest 0.49 -> no effect
  R2 | hr_physnet_v2 : R0 reports 10, lost 1, gained 0, moved >1 bpm 0, >2 bpm 0, largest 0.72 -> inconclusive
  R2 | hr_spectral   : R0 reports 10, lost 6, gained 0, moved >1 bpm 1, >2 bpm 0, largest 1.09 -> effect
  R3 | hr_physn

## 8. Check 2 - findings (2026-11-01)

**The pre-registered test is inconclusive.** Read through the web app's own path, PhysNet v2 reported 10 of the 12 recordings at R0: the framing gate refused one, as foreseen, and one kept too few confident windows even lossless. At R2, the app's recorder setting in H.264 (0.651 bits per pixel per frame; the files reached 0.656), nine of the ten were still reported, within 0.72 bpm of their R0 value (median move 0.23 bpm), and one lost its reading. One loss is too many for *no effect* and short of the two an *effect* needs. Part 0 fixed what each of those would do; an inconclusive outcome does neither.

**The lost reading was marginal before compression.** That recording kept 52% of its windows at R0, just above the 50% below which the app marks a capture degraded; at R2 it kept 14%, under the 25% needed to report. Compression costs v2 confident windows rather than moving its readings: its median window confidence falls from 0.874 at R0 to 0.794 at R2 and 0.747 at R4, while among the readings it reports, its median absolute error against the ECG rate stays at or below 0.46 bpm on every rung.

**Chroma subsampling alone costs nothing measurable** (H5: consistent). At R1, lossless 4:2:0, no reading was lost and none moved by more than 0.43 bpm (POS: 0.49).

**The other rungs decide nothing.** At half the rate (R3), v2 lost the same marginal recording and moved one reading by just over 1 bpm: inconclusive. At a quarter (R4, 0.163 bits per pixel per frame), it lost two and moved one by 2.1 bpm: an effect, at the rate where live captures had put the floor (0.1-0.2). VP8 at the app's rate (R5), the codec of a WebM recording, cost more than H.264 did: two lost and one moved by 1.4 bpm, an effect. H4 is unresolved: its R4 half holds, while its R2 half is inconclusive.

**POS is far more sensitive than v2.** It lost 6 of its 10 R0 readings at R2, 5 at R3 and R4, and 4 at R5, an effect on every lossy rung. Its median share of usable windows falls from 0.81 at R0 to 0.24 at R2, so its readings fail the confidence gate.

**Exploratory, not pre-registered: the losses fall mostly on Fitzpatrick types 4-6.** Each estimator reported five recordings of types 1-3 and five of types 4-6 at R0. At R2, POS lost all five of types 4-6 against one of types 1-3, and v2 lost one of types 4-6 and none of types 1-3; at R5, v2 lost two and none. In this subset, skin type and site nearly coincide: types 1-3 come mostly from one site group and types 4-6 mostly from another. The pattern therefore cannot be separated from the recording setup, and five recordings per group decide nothing.

**What it leaves.** On these twelve recordings, with an encoder that imitates a browser's but is not one, v2 at the app's recorder setting kept nine of ten readings within 0.72 bpm and lost one that was already marginal uncompressed. The "lossless lab video" caveat on the heart-rate figures stays, without the no-effect sentence Part 0 had ready. Any change - another recording rate or codec order, or a confidence rule for compressed input - would need its own rule, judged on data other than these twelve recordings.